# 3. Reconstruct Tree (NT) — local Simulated Bifurcation

This notebook replaces Amplify/Fujitsu DA4 with the unofficial `simulated-bifurcation` Python package and retains the existing nucleotide affinity construction, recursive Ncut selection, post-swap logic, output directory, Newick files, and reconstruction pickle keys.


In [ ]:
from __future__ import annotations

import os
import csv
import hashlib
import importlib.metadata
import json
import pickle
from dataclasses import dataclass
from pathlib import Path
from time import perf_counter
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import torch
import simulated_bifurcation as sb

In [ ]:
# ============================================================
# Project / paths
# ============================================================

def resolve_repository_root() -> Path:
    root = Path(
        os.environ.get(
            "PHYLOGENY_REPOSITORY_ROOT",
            Path.cwd(),
        )
    ).expanduser().resolve()

    if root.name in {"AA", "NT", "common"}:
        root = root.parent

    return root


REPOSITORY_ROOT = resolve_repository_root()
PROJECT_ROOT = REPOSITORY_ROOT / "NT"
DATA_ROOT = PROJECT_ROOT / "data"

MATRIX_DIR = DATA_ROOT / "matrices"
EVO_DISTANCE_DIR = DATA_ROOT / "evo_distances"
SIMULATION_MANIFEST = DATA_ROOT / "manifests" / "simulation_manifest.csv"

# Keep the same output root used by the previous reconstruction notebook.
OUT_DIR = PROJECT_ROOT / "results_reconstruct"


# ============================================================
# Simulated Bifurcation settings
# ============================================================

# Accuracy-oriented setting for n <= 30 on an NVIDIA H100.
# Run the preflight and one-tag smoke test before starting all Rep001--100 jobs.
SB_DEVICE = "cuda"
SB_DTYPE = "float64"
SB_AGENTS = 256
SB_MAX_STEPS = 5000
SB_MODE = "discrete"
SB_HEATED = False
SB_EARLY_STOPPING = False
SB_SAMPLING_PERIOD = 50
SB_CONVERGENCE_THRESHOLD = 50
SB_GLOBAL_SEED = 20260416
SB_SCALE_QUBO = True

# Cardinality-penalty schedule retained from the DA4 version.
ALPHA0 = 128.0
RETRIES = 6


# ============================================================
# Reconstruction settings
# ============================================================

KNN_K = 7
LAMBDA_SCALE = 1.0

POSTSWAP_MAX_PASSES = 20
POSTSWAP_TOL = 1e-12

EPS = 1e-12

# The result directory is initially empty. Keeping resume-safe checks is useful
# if a long Rep100 run is interrupted.
OVERWRITE = False
VERBOSE = False


# ============================================================
# Target selection
# ============================================================

# Direct tag selection has priority. Use one tag for the first smoke test.
RUN_TAGS: list[str] = [
    # "rtree_n30_bl0.125_rep001",
]

# If RUN_TAGS is empty, use the filters below.
INCLUDE_GENERATORS = {"rtree", "yule"}
INCLUDE_BLS = {"0.125", "0.250", "0.500", "0.625", "0.750"}
INCLUDE_REPS = set(range(1, 101))

RUN_VARIANTS = (
    "baseline_nmcut",
    "logkernel_nmcut",
    "jc69_selftune_nmcut",
    "logglobal_nmcut",
)

RUN_POSTSWAP_MODES = (
    "none",
    "ncut",
)

MAX_NEW_TAGS = None
SKIP_FULLY_DONE_TAGS = True


In [ ]:
@dataclass(frozen=True)
class ReconstructConfig:
    project_root: Path

    matrices_dir: Path
    evo_distance_dir: Path
    simulation_manifest: Path
    out_dir: Path

    variants: Tuple[str, ...]
    postswap_modes: Tuple[str, ...]

    sb_device: str
    sb_dtype: str
    sb_agents: int
    sb_max_steps: int
    sb_mode: str
    sb_heated: bool
    sb_early_stopping: bool
    sb_sampling_period: int
    sb_convergence_threshold: int
    sb_global_seed: int
    sb_scale_qubo: bool

    alpha0: float
    retries: int
    eps: float

    knn_k: int
    lambda_scale: float

    postswap_max_passes: int
    postswap_tol: float

    overwrite: bool
    verbose: bool


@dataclass
class TreeNode:
    name: Optional[str] = None
    index: Optional[int] = None
    left: Optional["TreeNode"] = None
    right: Optional["TreeNode"] = None

    @property
    def is_leaf(self) -> bool:
        return self.name is not None


def make_config() -> ReconstructConfig:
    return ReconstructConfig(
        project_root=PROJECT_ROOT,
        matrices_dir=MATRIX_DIR,
        evo_distance_dir=EVO_DISTANCE_DIR,
        simulation_manifest=SIMULATION_MANIFEST,
        out_dir=OUT_DIR,
        variants=tuple(RUN_VARIANTS),
        postswap_modes=tuple(RUN_POSTSWAP_MODES),
        sb_device=SB_DEVICE,
        sb_dtype=SB_DTYPE,
        sb_agents=SB_AGENTS,
        sb_max_steps=SB_MAX_STEPS,
        sb_mode=SB_MODE,
        sb_heated=SB_HEATED,
        sb_early_stopping=SB_EARLY_STOPPING,
        sb_sampling_period=SB_SAMPLING_PERIOD,
        sb_convergence_threshold=SB_CONVERGENCE_THRESHOLD,
        sb_global_seed=SB_GLOBAL_SEED,
        sb_scale_qubo=SB_SCALE_QUBO,
        alpha0=ALPHA0,
        retries=RETRIES,
        eps=EPS,
        knn_k=KNN_K,
        lambda_scale=LAMBDA_SCALE,
        postswap_max_passes=POSTSWAP_MAX_PASSES,
        postswap_tol=POSTSWAP_TOL,
        overwrite=OVERWRITE,
        verbose=VERBOSE,
    )


# ============================================================
# I/O helpers
# ============================================================

def ensure_dirs(
    cfg: ReconstructConfig,
) -> None:
    cfg.project_root.mkdir(
        parents=True,
        exist_ok=True,
    )

    cfg.out_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    for variant in cfg.variants:
        for mode in cfg.postswap_modes:
            (
                cfg.out_dir
                / variant
                / mode
            ).mkdir(
                parents=True,
                exist_ok=True,
            )


def read_simulation_manifest(
    path: Path,
) -> List[dict]:
    rows: List[dict] = []

    with path.open(newline="") as f:
        reader = csv.DictReader(f)

        for row in reader:
            status = (
                row.get("status")
                or ""
            ).lower()

            message = (
                row.get("message")
                or ""
            ).lower()

            if status == "ok" or message == "skipped_existing":
                rows.append(row)

    return rows


def load_pickle(
    path: Path,
) -> Dict[str, Any]:
    with path.open("rb") as f:
        return pickle.load(f)


def save_pickle(
    obj: Any,
    path: Path,
) -> None:
    with path.open("wb") as f:
        pickle.dump(obj, f)


def _csv_safe(value: Any) -> Any:
    if value is None:
        return ""
    if isinstance(value, (list, tuple, dict, np.ndarray)):
        if isinstance(value, np.ndarray):
            value = value.tolist()
        return json.dumps(value, ensure_ascii=False, separators=(",", ":"))
    if isinstance(value, (np.integer, np.floating, np.bool_)):
        return value.item()
    return value


def save_records_csv(
    records: List[dict],
    path: Path,
) -> None:
    """Save heterogeneous flat log records with a stable union of columns."""
    if not records:
        path.write_text("")
        return

    fieldnames: List[str] = []
    seen: set[str] = set()
    for record in records:
        for key in record:
            if key not in seen:
                seen.add(key)
                fieldnames.append(key)

    with path.open("w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        for record in records:
            writer.writerow({key: _csv_safe(record.get(key)) for key in fieldnames})


def outputs_exist(
    cfg: ReconstructConfig,
    tag: str,
    *,
    variant: str,
    postswap_mode: str,
) -> bool:
    out_dir = cfg.out_dir / variant / postswap_mode
    required = (
        out_dir / f"reconstruction_{tag}.pkl",
        out_dir / f"tree_{tag}.nwk",
        out_dir / f"solver_attempt_log_{tag}.csv",
        out_dir / f"cardinality_log_{tag}.csv",
        out_dir / f"node_log_{tag}.csv",
    )
    return all(path.exists() for path in required)


# ============================================================
# Tag selection helpers
# ============================================================

def parse_tag(
    tag: str,
) -> dict:
    # example: rtree_n30_bl0.125_rep001
    parts = tag.split("_")

    return {
        "generator": parts[0],
        "n_taxa": int(parts[1][1:]),
        "bl": parts[2][2:],
        "rep": int(parts[3][3:]),
    }


def should_select_tag(
    tag: str,
) -> bool:
    if RUN_TAGS:
        return tag in set(RUN_TAGS)

    info = parse_tag(tag)

    return (
        info["generator"] in INCLUDE_GENERATORS
        and info["bl"] in INCLUDE_BLS
        and info["rep"] in INCLUDE_REPS
    )


def fully_done_for_all_modes(
    cfg: ReconstructConfig,
    tag: str,
) -> bool:
    for variant in cfg.variants:
        for postswap_mode in cfg.postswap_modes:
            if not outputs_exist(
                cfg,
                tag,
                variant=variant,
                postswap_mode=postswap_mode,
            ):
                return False

    return True


def get_selected_tags(
    cfg: ReconstructConfig,
) -> List[str]:
    manifest_rows = read_simulation_manifest(
        cfg.simulation_manifest
    )

    all_tags = [
        row["tag"]
        for row in manifest_rows
    ]

    selected_tags = [
        tag
        for tag in all_tags
        if should_select_tag(tag)
    ]

    return selected_tags


def get_pending_tags(
    cfg: ReconstructConfig,
) -> List[str]:
    selected_tags = get_selected_tags(cfg)

    if SKIP_FULLY_DONE_TAGS:
        selected_tags = [
            tag
            for tag in selected_tags
            if not fully_done_for_all_modes(
                cfg,
                tag,
            )
        ]

    if MAX_NEW_TAGS is not None:
        selected_tags = selected_tags[:MAX_NEW_TAGS]

    return selected_tags


# ============================================================
# Matrix helpers
# ============================================================

def ensure_affinity_matrix(
    S: np.ndarray,
) -> np.ndarray:
    S = np.asarray(
        S,
        dtype=float,
    )

    S = 0.5 * (
        S
        + S.T
    )

    S = np.nan_to_num(
        S,
        nan=0.0,
        posinf=0.0,
        neginf=0.0,
    )

    S[S < 0] = 0.0

    np.fill_diagonal(
        S,
        0.0,
    )

    return S


def ensure_distance_matrix(
    D: np.ndarray,
) -> np.ndarray:
    D = np.asarray(
        D,
        dtype=float,
    )

    D = 0.5 * (
        D
        + D.T
    )

    D = np.nan_to_num(
        D,
        nan=0.0,
        posinf=0.0,
        neginf=0.0,
    )

    D[D < 0] = 0.0

    np.fill_diagonal(
        D,
        0.0,
    )

    return D


def kth_positive_or_fallback(
    values: np.ndarray,
    k: int,
    fallback: float,
) -> float:
    positive = np.sort(
        values[
            np.isfinite(values)
            & (values > 0)
        ]
    )

    if positive.size == 0:
        return fallback

    kk = min(
        k,
        positive.size,
    ) - 1

    return float(
        positive[kk]
    )


def self_tuning_affinity_from_distance(
    D: np.ndarray,
    *,
    knn_k: int,
    lambda_scale: float,
    eps: float,
) -> np.ndarray:
    """
    Self-tuning affinity:
        K_ij = exp(-D_ij^2 / (lambda * sigma_i * sigma_j))

    sigma_i is the k-th nearest positive distance from i.
    """
    D = ensure_distance_matrix(D)
    n = D.shape[0]

    off_diagonal = D[
        ~np.eye(
            n,
            dtype=bool,
        )
    ]

    positive = off_diagonal[
        np.isfinite(off_diagonal)
        & (off_diagonal > 0)
    ]

    global_fallback = (
        float(np.median(positive))
        if positive.size > 0
        else 1.0
    )

    sigmas = np.zeros(
        n,
        dtype=float,
    )

    for i in range(n):
        sigmas[i] = max(
            kth_positive_or_fallback(
                D[i],
                knn_k,
                global_fallback,
            ),
            eps,
        )

    denominator = (
        lambda_scale
        * np.outer(
            sigmas,
            sigmas,
        )
    )

    denominator = np.maximum(
        denominator,
        eps,
    )

    K = np.exp(
        -(
            D ** 2
        )
        / denominator
    )

    np.fill_diagonal(
        K,
        0.0,
    )

    return ensure_affinity_matrix(K)


def global_gaussian_affinity_from_distance(
    D: np.ndarray,
    *,
    lambda_scale: float,
    eps: float,
) -> np.ndarray:
    """
    Global-sigma Gaussian affinity:
        K_ij = exp(-D_ij^2 / (lambda * sigma^2))
    """
    D = ensure_distance_matrix(D)
    n = D.shape[0]

    off_diagonal = D[
        ~np.eye(
            n,
            dtype=bool,
        )
    ]

    positive = off_diagonal[
        np.isfinite(off_diagonal)
        & (off_diagonal > 0)
    ]

    sigma = (
        float(np.median(positive))
        if positive.size > 0
        else 1.0
    )

    sigma = max(
        sigma,
        eps,
    )

    denominator = max(
        lambda_scale
        * sigma
        * sigma,
        eps,
    )

    K = np.exp(
        -(
            D ** 2
        )
        / denominator
    )

    np.fill_diagonal(
        K,
        0.0,
    )

    return ensure_affinity_matrix(K)


def load_evo_distance_payload(
    cfg: ReconstructConfig,
    tag: str,
) -> Dict[str, Any]:
    path = (
        cfg.evo_distance_dir
        / f"{tag}_evo_distances.pkl"
    )

    if not path.exists():
        raise FileNotFoundError(
            f"Evolutionary distance file was not found: {path}"
        )

    return load_pickle(path)


def check_id_order(
    matrix_payload: Dict[str, Any],
    evo_payload: Dict[str, Any],
) -> None:
    ids1 = list(
        matrix_payload["ids"]
    )

    ids2 = list(
        evo_payload["ids"]
    )

    if ids1 != ids2:
        raise ValueError(
            "ID order mismatch between matrix payload and evolutionary-distance payload.\n"
            f"matrix ids[:5] = {ids1[:5]}\n"
            f"evo ids[:5]    = {ids2[:5]}"
        )


def build_affinity(
    payload: Dict[str, Any],
    variant: str,
    cfg: ReconstructConfig,
) -> np.ndarray:
    if variant == "baseline_nmcut":
        S = np.asarray(
            payload["paper_normbit_mean"],
            dtype=float,
        )

        return ensure_affinity_matrix(S)

    if variant == "logkernel_nmcut":
        D = np.asarray(
            payload["log_distance_from_mean"],
            dtype=float,
        )

        return self_tuning_affinity_from_distance(
            D,
            knn_k=cfg.knn_k,
            lambda_scale=cfg.lambda_scale,
            eps=cfg.eps,
        )

    if variant == "logglobal_nmcut":
        D = np.asarray(
            payload["log_distance_from_mean"],
            dtype=float,
        )

        return global_gaussian_affinity_from_distance(
            D,
            lambda_scale=cfg.lambda_scale,
            eps=cfg.eps,
        )

    if variant == "poisson20_selftune_nmcut":
        tag = payload["tag"]

        evo_payload = load_evo_distance_payload(
            cfg,
            tag,
        )

        check_id_order(
            payload,
            evo_payload,
        )

        D = np.asarray(
            evo_payload["poisson20_distance"],
            dtype=float,
        )

        return self_tuning_affinity_from_distance(
            D,
            knn_k=cfg.knn_k,
            lambda_scale=cfg.lambda_scale,
            eps=cfg.eps,
        )

    if variant == "wag_selftune_nmcut":
        tag = payload["tag"]

        evo_payload = load_evo_distance_payload(
            cfg,
            tag,
        )

        check_id_order(
            payload,
            evo_payload,
        )

        D = np.asarray(
            evo_payload["wag_distance"],
            dtype=float,
        )

        return self_tuning_affinity_from_distance(
            D,
            knn_k=cfg.knn_k,
            lambda_scale=cfg.lambda_scale,
            eps=cfg.eps,
        )

    if variant == "jc69_selftune_nmcut":
        tag = payload["tag"]

        evo_payload = load_evo_distance_payload(
            cfg,
            tag,
        )

        check_id_order(
            payload,
            evo_payload,
        )

        D = np.asarray(
            evo_payload["jc69_distance"],
            dtype=float,
        )

        return self_tuning_affinity_from_distance(
            D,
            knn_k=cfg.knn_k,
            lambda_scale=cfg.lambda_scale,
            eps=cfg.eps,
        )

    raise ValueError(
        f"unknown variant: {variant}"
    )


# ============================================================
# QUBO / cut metrics
# ============================================================

def torch_dtype_from_name(name: str) -> torch.dtype:
    mapping = {
        "float32": torch.float32,
        "float64": torch.float64,
    }
    if name not in mapping:
        raise ValueError(f"Unsupported SB_DTYPE: {name}")
    return mapping[name]


def synchronize_device(device: str) -> None:
    if str(device).startswith("cuda") and torch.cuda.is_available():
        torch.cuda.synchronize()


def stable_solver_seed(
    global_seed: int,
    *parts: Any,
) -> int:
    text = "|".join(str(x) for x in (global_seed, *parts))
    digest = hashlib.sha256(text.encode("utf-8")).digest()
    return int.from_bytes(digest[:8], "little") % (2**31 - 1)


def set_solver_seed(seed: int) -> None:
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def build_qubo_mincut(
    S: np.ndarray,
    c: int,
    alpha: float,
) -> Tuple[np.ndarray, float]:
    """
    Build an upper-triangular QUBO matrix for

        sum_{i<j} w_ij (x_i - x_j)^2
        + alpha (sum_i x_i - c)^2.

    The returned constant offset is alpha*c^2. The SB optimizer does not need
    the offset, but it is retained for objective-value verification and logs.
    """
    S = ensure_affinity_matrix(S)
    n = S.shape[0]

    Q = np.zeros((n, n), dtype=np.float64)

    # x_i^2 = x_i for binary variables.
    degrees = S.sum(axis=1)
    Q[np.diag_indices(n)] = degrees + alpha * (1.0 - 2.0 * c)

    for i in range(n - 1):
        for j in range(i + 1, n):
            Q[i, j] = -2.0 * S[i, j] + 2.0 * alpha

    offset = float(alpha * c * c)
    return Q, offset


def evaluate_qubo(
    Q_upper: np.ndarray,
    x_bin: np.ndarray,
    offset: float = 0.0,
) -> float:
    x = np.asarray(x_bin, dtype=np.float64)
    return float(x @ np.asarray(Q_upper, dtype=np.float64) @ x + offset)


def _candidate_matrix(
    vectors: torch.Tensor,
    n_variables: int,
) -> np.ndarray:
    arr = vectors.detach().to("cpu").numpy()

    if arr.ndim == 1:
        if arr.size != n_variables:
            raise ValueError(f"Unexpected SB vector shape: {arr.shape}")
        return arr.reshape(1, n_variables)

    if arr.ndim != 2:
        if arr.size % n_variables != 0:
            raise ValueError(f"Unexpected SB vectors shape: {arr.shape}")
        return arr.reshape(-1, n_variables)

    if arr.shape[1] == n_variables:
        return arr
    if arr.shape[0] == n_variables:
        return arr.T

    raise ValueError(f"Unexpected SB vectors shape: {arr.shape}")


def solve_qubo_sb(
    Q_upper: np.ndarray,
    *,
    offset: float,
    target_cardinality: int,
    cfg: ReconstructConfig,
    seed: int,
) -> Dict[str, Any]:
    """Solve one QUBO and select the best feasible vector among all agents."""
    Q_original = np.asarray(Q_upper, dtype=np.float64)
    if Q_original.ndim != 2 or Q_original.shape[0] != Q_original.shape[1]:
        raise ValueError("Q_upper must be square.")
    if not np.allclose(Q_original, np.triu(Q_original)):
        raise ValueError("Q_upper must be upper triangular.")

    scale = float(np.max(np.abs(Q_original))) if Q_original.size else 1.0
    if not cfg.sb_scale_qubo or not np.isfinite(scale) or scale <= 0.0:
        scale = 1.0
    Q_solver = Q_original / scale

    device = cfg.sb_device
    dtype = torch_dtype_from_name(cfg.sb_dtype)
    q_tensor = torch.as_tensor(Q_solver, dtype=dtype, device=device)

    set_solver_seed(seed)
    synchronize_device(device)
    start = perf_counter()

    try:
        vectors_tensor, values_tensor = sb.minimize(
            q_tensor,
            domain="binary",
            device=device,
            agents=cfg.sb_agents,
            max_steps=cfg.sb_max_steps,
            best_only=False,
            mode=cfg.sb_mode,
            heated=cfg.sb_heated,
            verbose=False,
            early_stopping=cfg.sb_early_stopping,
            sampling_period=cfg.sb_sampling_period,
            convergence_threshold=cfg.sb_convergence_threshold,
        )
        synchronize_device(device)
        solver_time_sec = perf_counter() - start

        candidates = _candidate_matrix(vectors_tensor, Q_original.shape[0])
        candidates = np.rint(candidates).clip(0, 1).astype(np.int8)

        objectives = np.array(
            [evaluate_qubo(Q_original, x, offset) for x in candidates],
            dtype=np.float64,
        )
        cardinalities = candidates.sum(axis=1).astype(int)
        feasible_mask = cardinalities == int(target_cardinality)
        feasible_indices = np.where(feasible_mask)[0]

        if feasible_indices.size:
            chosen = int(feasible_indices[np.argmin(objectives[feasible_indices])])
            feasible = True
        else:
            chosen = int(np.argmin(objectives))
            feasible = False

        values = values_tensor.detach().to("cpu").numpy().reshape(-1)
        returned_value = (
            float(values[chosen])
            if chosen < values.size
            else float("nan")
        )

        x = candidates[chosen].copy()
        return {
            "solver_success": True,
            "feasible": bool(feasible),
            "solution": x,
            "solution_cardinality": int(x.sum()),
            "target_cardinality": int(target_cardinality),
            "num_agents_returned": int(candidates.shape[0]),
            "num_feasible_agents": int(feasible_mask.sum()),
            "solver_time_sec": float(solver_time_sec),
            "seed": int(seed),
            "qubo_scale": float(scale),
            "returned_objective_scaled_without_offset": returned_value,
            "recalculated_objective": float(objectives[chosen]),
            "solution_bits": "".join(str(int(v)) for v in x),
            "error": None,
        }

    except Exception as exc:
        synchronize_device(device)
        solver_time_sec = perf_counter() - start
        return {
            "solver_success": False,
            "feasible": False,
            "solution": None,
            "solution_cardinality": None,
            "target_cardinality": int(target_cardinality),
            "num_agents_returned": 0,
            "num_feasible_agents": 0,
            "solver_time_sec": float(solver_time_sec),
            "seed": int(seed),
            "qubo_scale": float(scale),
            "returned_objective_scaled_without_offset": float("nan"),
            "recalculated_objective": float("nan"),
            "solution_bits": "",
            "error": f"{type(exc).__name__}: {exc}",
        }


def compute_mincut_value(
    S: np.ndarray,
    x_bin: np.ndarray,
) -> float:
    A = np.where(
        x_bin == 1
    )[0]

    B = np.where(
        x_bin == 0
    )[0]

    if len(A) == 0 or len(B) == 0:
        return float("inf")

    return float(
        S[np.ix_(A, B)].sum()
    )


def compute_assoc_terms(
    S: np.ndarray,
    x_bin: np.ndarray,
) -> Tuple[float, float]:
    A = np.where(
        x_bin == 1
    )[0]

    B = np.where(
        x_bin == 0
    )[0]

    assoc_A = (
        float(S[A, :].sum())
        if len(A)
        else 0.0
    )

    assoc_B = (
        float(S[B, :].sum())
        if len(B)
        else 0.0
    )

    return assoc_A, assoc_B


def compute_ncut(
    S: np.ndarray,
    x_bin: np.ndarray,
    eps: float,
) -> Tuple[float, float, float, float]:
    mincut_value = compute_mincut_value(
        S,
        x_bin,
    )

    assoc_A, assoc_B = compute_assoc_terms(
        S,
        x_bin,
    )

    denominator_A = max(
        assoc_A,
        eps,
    )

    denominator_B = max(
        assoc_B,
        eps,
    )

    ncut = float(
        mincut_value / denominator_A
        + mincut_value / denominator_B
    )

    return (
        ncut,
        mincut_value,
        assoc_A,
        assoc_B,
    )


# ============================================================
# Ncut bit-swap post-processing
# ============================================================

def ncut_bit_swap_postprocess(
    S: np.ndarray,
    x_bin: np.ndarray,
    *,
    eps: float,
    max_passes: int,
    tol: float,
    verbose: bool = False,
) -> Dict[str, Any]:
    """
    Preserve cardinality by swapping one element from A and one from B.
    Improvement target is Ncut.
    """
    S = ensure_affinity_matrix(S)

    x = np.asarray(
        x_bin,
        dtype=int,
    ).copy()

    def evaluate(
        z: np.ndarray,
    ) -> Dict[str, float]:
        ncut, mincut_value, assoc_A, assoc_B = compute_ncut(
            S,
            z,
            eps,
        )

        return {
            "ncut": float(ncut),
            "mincut": float(mincut_value),
            "assoc_A": float(assoc_A),
            "assoc_B": float(assoc_B),
        }

    current = evaluate(x)
    history: List[dict] = []

    for pass_index in range(
        1,
        max_passes + 1,
    ):
        A = np.where(
            x == 1
        )[0]

        B = np.where(
            x == 0
        )[0]

        best_pair = None
        best_eval = current

        for i in A:
            for j in B:
                y = x.copy()
                y[i] = 0
                y[j] = 1

                candidate = evaluate(y)

                if candidate["ncut"] < best_eval["ncut"] - tol:
                    best_pair = (
                        int(i),
                        int(j),
                    )

                    best_eval = candidate

        if best_pair is None:
            break

        i, j = best_pair

        x[i] = 0
        x[j] = 1

        current = best_eval

        history.append(
            {
                "pass": pass_index,
                "swap_out_from_A": i,
                "swap_in_from_B": j,
                "ncut_after": current["ncut"],
                "mincut_after": current["mincut"],
                "assoc_A_after": current["assoc_A"],
                "assoc_B_after": current["assoc_B"],
            }
        )

        if verbose:
            print(
                f"[postswap:ncut] pass={pass_index} "
                f"swap(A:{i} <-> B:{j}) "
                f"ncut={current['ncut']:.6f} "
                f"mincut={current['mincut']:.6f}"
            )

    return {
        "x_bin": x.copy(),
        "ncut": current["ncut"],
        "mincut": current["mincut"],
        "assoc_A": current["assoc_A"],
        "assoc_B": current["assoc_B"],
        "num_improving_swaps": len(history),
        "history": history,
    }


# ============================================================
# Solve one split
# ============================================================

def cut_once(
    S: np.ndarray,
    cfg: ReconstructConfig,
    *,
    node_id: str,
    tag: str,
    variant: str,
    postswap_mode: str,
    verbose: bool = False,
) -> Dict[str, Any]:
    S = ensure_affinity_matrix(S)
    n = S.shape[0]

    if n < 2:
        return {
            "status": "trivial",
            "solver_attempts": [],
            "cardinality_results": [],
        }

    best: Dict[str, Any] = {
        "status": "fail",
        "ncut": float("inf"),
    }
    solver_attempts: List[dict] = []
    cardinality_results: List[dict] = []

    for c in range(1, n // 2 + 1):
        alpha = float(cfg.alpha0)
        accepted: Optional[dict] = None
        last_alpha = alpha

        for attempt in range(1, cfg.retries + 1):
            last_alpha = alpha
            Q_upper, offset = build_qubo_mincut(S, c=c, alpha=alpha)
            seed = stable_solver_seed(
                cfg.sb_global_seed,
                tag,
                node_id,
                n,
                c,
                attempt,
            )

            solved = solve_qubo_sb(
                Q_upper,
                offset=offset,
                target_cardinality=c,
                cfg=cfg,
                seed=seed,
            )

            attempt_record = {
                "tag": tag,
                "variant": variant,
                "postswap_mode": postswap_mode,
                "node_id": node_id,
                "node_size": int(n),
                "target_cardinality": int(c),
                "attempt": int(attempt),
                "retry_count_so_far": int(attempt - 1),
                "alpha": float(alpha),
                **{k: v for k, v in solved.items() if k != "solution"},
            }
            solver_attempts.append(attempt_record)

            if not solved["solver_success"] or not solved["feasible"]:
                alpha *= 2.0
                continue

            x_before = np.asarray(solved["solution"], dtype=int)
            ncut_before, mincut_before, assoc_A_before, assoc_B_before = compute_ncut(
                S,
                x_before,
                cfg.eps,
            )

            postswap_info = None
            x_after = x_before.copy()
            ncut_after = float(ncut_before)
            mincut_after = float(mincut_before)
            assoc_A_after = float(assoc_A_before)
            assoc_B_after = float(assoc_B_before)

            if postswap_mode == "ncut":
                postswap_info = ncut_bit_swap_postprocess(
                    S,
                    x_before,
                    eps=cfg.eps,
                    max_passes=cfg.postswap_max_passes,
                    tol=cfg.postswap_tol,
                    verbose=verbose,
                )
                x_after = postswap_info["x_bin"]
                ncut_after = float(postswap_info["ncut"])
                mincut_after = float(postswap_info["mincut"])
                assoc_A_after = float(postswap_info["assoc_A"])
                assoc_B_after = float(postswap_info["assoc_B"])
            elif postswap_mode != "none":
                raise ValueError(f"unknown postswap_mode: {postswap_mode}")

            accepted = {
                "tag": tag,
                "variant": variant,
                "postswap_mode": postswap_mode,
                "node_id": node_id,
                "node_size": int(n),
                "target_cardinality": int(c),
                "feasible": True,
                "attempt_count": int(attempt),
                "retry_count": int(attempt - 1),
                "accepted_alpha": float(alpha),
                "final_alpha": float(alpha),
                "solver_time_sec_total": float(
                    sum(
                        row["solver_time_sec"]
                        for row in solver_attempts
                        if row["node_id"] == node_id
                        and row["target_cardinality"] == c
                    )
                ),
                "ncut_before_postswap": float(ncut_before),
                "ncut_after_postswap": float(ncut_after),
                "mincut_before_postswap": float(mincut_before),
                "mincut_after_postswap": float(mincut_after),
                "assoc_A_before_postswap": float(assoc_A_before),
                "assoc_B_before_postswap": float(assoc_B_before),
                "assoc_A_after_postswap": float(assoc_A_after),
                "assoc_B_after_postswap": float(assoc_B_after),
                "num_postswaps": int(
                    postswap_info["num_improving_swaps"]
                    if postswap_info is not None
                    else 0
                ),
                "solution_bits_before_postswap": "".join(str(int(v)) for v in x_before),
                "solution_bits_after_postswap": "".join(str(int(v)) for v in x_after),
                "selected": False,
            }
            cardinality_results.append(accepted)

            if verbose:
                print(
                    f"[cut] node={node_id} c={c} attempt={attempt} "
                    f"alpha={alpha:.1f} ncut={ncut_after:.6f}"
                )

            if ncut_after < best["ncut"]:
                best = {
                    "status": "ok",
                    "c": int(c),
                    "k": int(x_after.sum()),
                    "alpha": float(alpha),
                    "retry": int(attempt),
                    "retry_count": int(attempt - 1),
                    "x_bin": x_after.copy(),
                    "ncut": float(ncut_after),
                    "mincut": float(mincut_after),
                    "assoc_A": float(assoc_A_after),
                    "assoc_B": float(assoc_B_after),
                    "postswap_mode": postswap_mode,
                    "postswap": postswap_info,
                }
            break

        if accepted is None:
            attempts_for_c = [
                row
                for row in solver_attempts
                if row["node_id"] == node_id
                and row["target_cardinality"] == c
            ]
            cardinality_results.append(
                {
                    "tag": tag,
                    "variant": variant,
                    "postswap_mode": postswap_mode,
                    "node_id": node_id,
                    "node_size": int(n),
                    "target_cardinality": int(c),
                    "feasible": False,
                    "attempt_count": int(len(attempts_for_c)),
                    "retry_count": int(max(0, len(attempts_for_c) - 1)),
                    "accepted_alpha": None,
                    "final_alpha": float(last_alpha),
                    "solver_time_sec_total": float(
                        sum(row["solver_time_sec"] for row in attempts_for_c)
                    ),
                    "ncut_before_postswap": None,
                    "ncut_after_postswap": None,
                    "mincut_before_postswap": None,
                    "mincut_after_postswap": None,
                    "assoc_A_before_postswap": None,
                    "assoc_B_before_postswap": None,
                    "assoc_A_after_postswap": None,
                    "assoc_B_after_postswap": None,
                    "num_postswaps": None,
                    "solution_bits_before_postswap": "",
                    "solution_bits_after_postswap": "",
                    "selected": False,
                }
            )

    if best["status"] == "ok":
        for record in cardinality_results:
            record["selected"] = (
                record["feasible"]
                and record["target_cardinality"] == best["c"]
            )

    best["solver_attempts"] = solver_attempts
    best["cardinality_results"] = cardinality_results
    return best


# ============================================================
# Fallback split
# ============================================================

def spectral_fallback_split(
    S: np.ndarray,
    eps: float,
) -> Tuple[np.ndarray, np.ndarray]:
    S = ensure_affinity_matrix(S)
    n = S.shape[0]

    if n == 2:
        return (
            np.array(
                [0],
                dtype=int,
            ),
            np.array(
                [1],
                dtype=int,
            ),
        )

    degree = S.sum(axis=1)

    if np.all(degree <= eps):
        order = np.arange(
            n,
            dtype=int,
        )

        k = max(
            1,
            n // 2,
        )

        return (
            order[:k],
            order[k:],
        )

    Dinv = np.diag(
        1.0
        / np.sqrt(
            np.maximum(
                degree,
                eps,
            )
        )
    )

    L = (
        np.diag(degree)
        - S
    )

    Lsym = (
        Dinv
        @ L
        @ Dinv
    )

    try:
        _, eigvecs = np.linalg.eigh(
            Lsym
        )

        vec = (
            eigvecs[:, 1]
            if n >= 2
            else np.arange(
                n,
                dtype=float,
            )
        )

    except np.linalg.LinAlgError:
        vec = np.arange(
            n,
            dtype=float,
        )

    order = np.argsort(
        vec,
        kind="mergesort",
    )

    k = max(
        1,
        n // 2,
    )

    A = order[:k]
    B = order[k:]

    if len(A) == 0 or len(B) == 0:
        order = np.arange(
            n,
            dtype=int,
        )

        k = max(
            1,
            n // 2,
        )

        A = order[:k]
        B = order[k:]

    return (
        A.astype(int),
        B.astype(int),
    )


# ============================================================
# Recursive tree reconstruction
# ============================================================

def reconstruct_subtree(
    S_all: np.ndarray,
    ids: List[str],
    indices: np.ndarray,
    cfg: ReconstructConfig,
    *,
    tag: str,
    variant: str,
    postswap_mode: str,
    node_id: str,
    cut_log: List[dict],
    solver_attempt_log: List[dict],
    cardinality_log: List[dict],
) -> TreeNode:
    n = len(indices)

    if n == 1:
        idx = int(indices[0])
        return TreeNode(name=ids[idx], index=idx)

    if n == 2:
        return TreeNode(
            left=TreeNode(name=ids[int(indices[0])], index=int(indices[0])),
            right=TreeNode(name=ids[int(indices[1])], index=int(indices[1])),
        )

    S = ensure_affinity_matrix(S_all[np.ix_(indices, indices)])

    best = cut_once(
        S,
        cfg,
        node_id=node_id,
        tag=tag,
        variant=variant,
        postswap_mode=postswap_mode,
        verbose=cfg.verbose,
    )

    node_members = [ids[int(i)] for i in indices]
    local_attempts = best.get("solver_attempts", [])
    local_cardinalities = best.get("cardinality_results", [])
    for record in local_attempts:
        record["node_members"] = node_members
    for record in local_cardinalities:
        record["node_members"] = node_members

    fallback_used = False
    local_ncut = None
    c_used = None
    selected_alpha = None
    selected_retry_count = None
    num_postswaps = None

    if best.get("status") != "ok":
        fallback_used = True
        A_local, B_local = spectral_fallback_split(S, cfg.eps)
    else:
        x = best["x_bin"]
        A_local = np.where(x == 1)[0]
        B_local = np.where(x == 0)[0]

        if len(A_local) == 0 or len(B_local) == 0:
            fallback_used = True
            A_local, B_local = spectral_fallback_split(S, cfg.eps)
        else:
            local_ncut = float(best["ncut"])
            c_used = int(best["c"])
            selected_alpha = float(best["alpha"])
            selected_retry_count = int(best["retry_count"])
            num_postswaps = int(
                best["postswap"]["num_improving_swaps"]
                if best.get("postswap") is not None
                else 0
            )

    A = np.sort(indices[A_local])
    B = np.sort(indices[B_local])

    if len(A) == 0 or len(B) == 0:
        order = np.sort(indices)
        k = max(1, len(order) // 2)
        A = order[:k]
        B = order[k:]
        fallback_used = True
        local_ncut = None
        c_used = None
        selected_alpha = None
        selected_retry_count = None
        num_postswaps = None

    for record in local_cardinalities:
        record["node_fallback_used"] = bool(fallback_used)
    solver_attempt_log.extend(local_attempts)
    cardinality_log.extend(local_cardinalities)

    cut_log.append(
        {
            "tag": tag,
            "variant": variant,
            "postswap_mode": postswap_mode,
            "node_id": node_id,
            "node_size": int(n),
            "A_size": int(len(A)),
            "B_size": int(len(B)),
            "ncut": local_ncut,
            "c": c_used,
            "selected_alpha": selected_alpha,
            "selected_retry_count": selected_retry_count,
            "fallback_used": bool(fallback_used),
            "num_postswaps": num_postswaps,
            "solver_call_count": int(len(local_attempts)),
            "solver_time_sec": float(sum(x["solver_time_sec"] for x in local_attempts)),
            "feasible_cardinality_count": int(
                sum(bool(x["feasible"]) for x in local_cardinalities)
            ),
            "failed_cardinality_count": int(
                sum(not bool(x["feasible"]) for x in local_cardinalities)
            ),
            "members_A": [ids[int(i)] for i in A],
            "members_B": [ids[int(i)] for i in B],
        }
    )

    left = reconstruct_subtree(
        S_all=S_all,
        ids=ids,
        indices=A,
        cfg=cfg,
        tag=tag,
        variant=variant,
        postswap_mode=postswap_mode,
        node_id=f"{node_id}0",
        cut_log=cut_log,
        solver_attempt_log=solver_attempt_log,
        cardinality_log=cardinality_log,
    )

    right = reconstruct_subtree(
        S_all=S_all,
        ids=ids,
        indices=B,
        cfg=cfg,
        tag=tag,
        variant=variant,
        postswap_mode=postswap_mode,
        node_id=f"{node_id}1",
        cut_log=cut_log,
        solver_attempt_log=solver_attempt_log,
        cardinality_log=cardinality_log,
    )

    return TreeNode(left=left, right=right)


# ============================================================
# Tree serialization / split extraction
# ============================================================

def to_newick(
    node: TreeNode,
) -> str:
    if node.is_leaf:
        return str(
            node.name
        )

    assert node.left is not None
    assert node.right is not None

    return (
        f"({to_newick(node.left)},"
        f"{to_newick(node.right)})"
    )


def collect_leaf_indices(
    node: TreeNode,
) -> set[int]:
    if node.is_leaf:
        return {
            int(node.index)
        }

    assert node.left is not None
    assert node.right is not None

    return (
        collect_leaf_indices(node.left)
        | collect_leaf_indices(node.right)
    )


def enumerate_internal_splits(
    node: TreeNode,
    all_leaves: set[int],
    acc: set[frozenset[int]],
) -> set[int]:
    if node.is_leaf:
        return {
            int(node.index)
        }

    assert node.left is not None
    assert node.right is not None

    left = enumerate_internal_splits(
        node.left,
        all_leaves,
        acc,
    )

    right = enumerate_internal_splits(
        node.right,
        all_leaves,
        acc,
    )

    here = (
        left
        | right
    )

    if 0 < len(here) < len(all_leaves):
        complement = (
            all_leaves
            - here
        )

        small = (
            here
            if len(here) <= len(complement)
            else complement
        )

        if 2 <= len(small) <= len(all_leaves) - 2:
            acc.add(
                frozenset(
                    sorted(small)
                )
            )

    return here


def tree_to_internal_splits(
    node: TreeNode,
) -> List[List[int]]:
    all_leaves = collect_leaf_indices(
        node
    )

    acc: set[frozenset[int]] = set()

    enumerate_internal_splits(
        node,
        all_leaves,
        acc,
    )

    return [
        list(split)
        for split in sorted(
            acc,
            key=lambda x: (
                len(x),
                tuple(x),
            ),
        )
    ]


# ============================================================
# Run one dataset
# ============================================================

def reconstruct_one_tag(
    cfg: ReconstructConfig,
    tag: str,
    *,
    variant: str,
    postswap_mode: str,
) -> Dict[str, Any]:
    matrix_path = cfg.matrices_dir / f"{tag}_matrices.pkl"
    if not matrix_path.exists():
        raise FileNotFoundError(matrix_path)

    payload = load_pickle(matrix_path)
    ids = list(payload["ids"])
    S_all = build_affinity(payload, variant, cfg)

    cut_log: List[dict] = []
    solver_attempt_log: List[dict] = []
    cardinality_log: List[dict] = []

    synchronize_device(cfg.sb_device)
    reconstruction_start = perf_counter()

    root = reconstruct_subtree(
        S_all=S_all,
        ids=ids,
        indices=np.arange(len(ids), dtype=int),
        cfg=cfg,
        tag=tag,
        variant=variant,
        postswap_mode=postswap_mode,
        node_id="R",
        cut_log=cut_log,
        solver_attempt_log=solver_attempt_log,
        cardinality_log=cardinality_log,
    )

    synchronize_device(cfg.sb_device)
    reconstruction_wall_time_sec = perf_counter() - reconstruction_start

    newick = to_newick(root) + ";"
    splits = tree_to_internal_splits(root)

    fallback_count = sum(int(x["fallback_used"]) for x in cut_log)
    total_postswaps = sum(
        int(x["num_postswaps"])
        for x in cut_log
        if x["num_postswaps"] is not None
    )
    ncut_values = [x["ncut"] for x in cut_log if x["ncut"] is not None]
    total_ncut = float(np.sum(ncut_values)) if ncut_values else None

    solver_time_sec_total = float(
        sum(x["solver_time_sec"] for x in solver_attempt_log)
    )
    feasible_cardinality_count = int(
        sum(bool(x["feasible"]) for x in cardinality_log)
    )
    failed_cardinality_count = int(
        sum(not bool(x["feasible"]) for x in cardinality_log)
    )

    out = {
        "tag": tag,
        "variant": variant,
        "postswap_mode": postswap_mode,
        "n": len(ids),
        "newick": newick,
        "internal_splits": splits,
        "total_ncut": total_ncut,
        "num_internal_splits": len(splits),
        "num_cut_nodes": len(cut_log),
        "fallback_count": fallback_count,
        "total_postswaps": total_postswaps,
        "solver_time_sec_total": solver_time_sec_total,
        "reconstruction_wall_time_sec": float(reconstruction_wall_time_sec),
        "solver_call_count": len(solver_attempt_log),
        "feasible_cardinality_count": feasible_cardinality_count,
        "failed_cardinality_count": failed_cardinality_count,
        "cut_log": cut_log,
        "solver_attempt_log": solver_attempt_log,
        "cardinality_log": cardinality_log,
        "meta": {
            "solver_backend": "simulated-bifurcation",
            "simulated_bifurcation_version": importlib.metadata.version(
                "simulated-bifurcation"
            ),
            "torch_version": torch.__version__,
            "cuda_available": torch.cuda.is_available(),
            "gpu_name": (
                torch.cuda.get_device_name(0)
                if torch.cuda.is_available()
                else None
            ),
            "sb_device": cfg.sb_device,
            "sb_dtype": cfg.sb_dtype,
            "sb_agents": cfg.sb_agents,
            "sb_max_steps": cfg.sb_max_steps,
            "sb_best_only": False,
            "sb_mode": cfg.sb_mode,
            "sb_heated": cfg.sb_heated,
            "sb_early_stopping": cfg.sb_early_stopping,
            "sb_sampling_period": cfg.sb_sampling_period,
            "sb_convergence_threshold": cfg.sb_convergence_threshold,
            "sb_global_seed": cfg.sb_global_seed,
            "sb_scale_qubo": cfg.sb_scale_qubo,
            "alpha0": cfg.alpha0,
            "retries": cfg.retries,
            "knn_k": cfg.knn_k,
            "lambda_scale": cfg.lambda_scale,
            "postswap_mode": postswap_mode,
            "postswap_max_passes": cfg.postswap_max_passes,
        },
    }

    out_dir = cfg.out_dir / variant / postswap_mode
    out_dir.mkdir(parents=True, exist_ok=True)

    (out_dir / f"tree_{tag}.nwk").write_text(newick)
    save_records_csv(
        solver_attempt_log,
        out_dir / f"solver_attempt_log_{tag}.csv",
    )
    save_records_csv(
        cardinality_log,
        out_dir / f"cardinality_log_{tag}.csv",
    )
    save_records_csv(
        cut_log,
        out_dir / f"node_log_{tag}.csv",
    )
    # Write the reconstruction pickle last; it acts as the completion marker.
    save_pickle(out, out_dir / f"reconstruction_{tag}.pkl")

    return out





In [ ]:
# ============================================================
# Preflight checks
# ============================================================

def print_environment_summary(cfg: ReconstructConfig) -> None:
    print("simulated-bifurcation:", importlib.metadata.version("simulated-bifurcation"))
    print("torch:", torch.__version__)
    print("CUDA available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("GPU:", torch.cuda.get_device_name(0))
        free_bytes, total_bytes = torch.cuda.mem_get_info()
        print(f"GPU memory free/total: {free_bytes / 2**30:.1f}/{total_bytes / 2**30:.1f} GiB")
    print("SB settings:", {
        "device": cfg.sb_device,
        "dtype": cfg.sb_dtype,
        "agents": cfg.sb_agents,
        "max_steps": cfg.sb_max_steps,
        "mode": cfg.sb_mode,
        "heated": cfg.sb_heated,
        "early_stopping": cfg.sb_early_stopping,
    })


def validate_qubo_expansion() -> None:
    S = np.array(
        [
            [0.0, 10.0, 1.0, 1.0],
            [10.0, 0.0, 1.0, 1.0],
            [1.0, 1.0, 0.0, 10.0],
            [1.0, 1.0, 10.0, 0.0],
        ],
        dtype=float,
    )
    c = 2
    alpha = 128.0
    Q, offset = build_qubo_mincut(S, c=c, alpha=alpha)

    for mask in range(2 ** len(S)):
        x = np.array([(mask >> i) & 1 for i in range(len(S))], dtype=int)
        direct = sum(S[i, j] * (x[i] - x[j]) ** 2 for i in range(len(S) - 1) for j in range(i + 1, len(S))) + alpha * (int(x.sum()) - c) ** 2
        expanded = evaluate_qubo(Q, x, offset)
        if not np.isclose(direct, expanded, atol=1e-9):
            raise AssertionError(
                f"QUBO expansion mismatch: x={x}, direct={direct}, expanded={expanded}"
            )
    print("[ok] QUBO expansion matches the original objective for all 16 vectors.")


def warmup_simulated_bifurcation(cfg: ReconstructConfig) -> float:
    Q = np.array([[-1.0, 2.0], [0.0, -1.0]], dtype=float)
    result = solve_qubo_sb(
        Q,
        offset=0.0,
        target_cardinality=1,
        cfg=cfg,
        seed=stable_solver_seed(cfg.sb_global_seed, "warmup"),
    )
    if not result["solver_success"] or not result["feasible"]:
        raise RuntimeError(f"SB warm-up failed: {result}")
    if not np.isclose(result["recalculated_objective"], -1.0, atol=1e-8):
        raise RuntimeError(f"Unexpected warm-up objective: {result}")
    print(f"[ok] SB warm-up: {result['solver_time_sec']:.3f} sec")
    return float(result["solver_time_sec"])


def validate_sb_cardinality_qubo(cfg: ReconstructConfig) -> None:
    S = np.array(
        [
            [0.0, 10.0, 1.0, 1.0],
            [10.0, 0.0, 1.0, 1.0],
            [1.0, 1.0, 0.0, 10.0],
            [1.0, 1.0, 10.0, 0.0],
        ],
        dtype=float,
    )
    c = 2
    Q, offset = build_qubo_mincut(S, c=c, alpha=cfg.alpha0)
    result = solve_qubo_sb(
        Q,
        offset=offset,
        target_cardinality=c,
        cfg=cfg,
        seed=stable_solver_seed(cfg.sb_global_seed, "preflight-cardinality"),
    )
    if not result["solver_success"] or not result["feasible"]:
        raise RuntimeError(f"SB cardinality test failed: {result}")
    mincut = compute_mincut_value(S, result["solution"])
    if not np.isclose(mincut, 4.0, atol=1e-8):
        raise RuntimeError(
            f"SB did not recover the known minimum cut in the preflight test: {result}"
        )
    print(
        "[ok] SB recovered the known c=2 minimum cut; "
        f"time={result['solver_time_sec']:.3f} sec, "
        f"feasible agents={result['num_feasible_agents']}/{result['num_agents_returned']}"
    )


def check_input_files(cfg: ReconstructConfig, tags: List[str]) -> None:
    missing_matrices = [
        tag for tag in tags
        if not (cfg.matrices_dir / f"{tag}_matrices.pkl").exists()
    ]
    needs_evo = any(
        variant in {"poisson20_selftune_nmcut", "wag_selftune_nmcut", "jc69_selftune_nmcut"}
        for variant in cfg.variants
    )
    missing_evo = []
    if needs_evo:
        missing_evo = [
            tag for tag in tags
            if not (cfg.evo_distance_dir / f"{tag}_evo_distances.pkl").exists()
        ]

    if missing_matrices or missing_evo:
        raise FileNotFoundError(
            f"missing matrix files={len(missing_matrices)}, "
            f"missing evolutionary-distance files={len(missing_evo)}; "
            f"examples={missing_matrices[:3] + missing_evo[:3]}"
        )
    print(f"[ok] input files found for {len(tags)} selected tags.")


def inspect_affinity_variants(cfg: ReconstructConfig, tag: str) -> None:
    payload = load_pickle(cfg.matrices_dir / f"{tag}_matrices.pkl")
    n = len(payload["ids"])
    print(f"[preflight affinity] tag={tag}, n={n}")
    for variant in cfg.variants:
        S = build_affinity(payload, variant, cfg)
        if S.shape != (n, n):
            raise ValueError(f"Unexpected shape for {variant}: {S.shape}")
        if not np.all(np.isfinite(S)):
            raise ValueError(f"Non-finite affinity values for {variant}")
        if np.any(S < 0) or not np.allclose(S, S.T):
            raise ValueError(f"Invalid affinity matrix for {variant}")
        offdiag = S[~np.eye(n, dtype=bool)]
        print(
            f"  {variant:30s} min={offdiag.min():.6g} "
            f"median={np.median(offdiag):.6g} max={offdiag.max():.6g}"
        )


In [ ]:
cfg = make_config()
ensure_dirs(cfg)

if cfg.sb_device.startswith("cuda") and not torch.cuda.is_available():
    raise RuntimeError("SB_DEVICE is cuda, but CUDA is not available.")

print_environment_summary(cfg)
validate_qubo_expansion()
warmup_simulated_bifurcation(cfg)
validate_sb_cardinality_qubo(cfg)

pending_tags = get_pending_tags(cfg)
print(f"[info] target tags: {len(pending_tags)}")
print("[info] example:", pending_tags[:10])
print("[info] variants:", cfg.variants)
print("[info] postswap modes:", cfg.postswap_modes)

check_input_files(cfg, pending_tags)
if pending_tags:
    inspect_affinity_variants(cfg, pending_tags[0])


In [ ]:
# ============================================================
# Reconstruction
# ============================================================
# For the first run, set RUN_TAGS to one representative tag and rerun the
# settings/functions/preflight cells before executing this cell.

for tag in pending_tags:
    matrix_path = cfg.matrices_dir / f"{tag}_matrices.pkl"

    for variant in cfg.variants:
        for postswap_mode in cfg.postswap_modes:
            if (
                not cfg.overwrite
                and outputs_exist(
                    cfg,
                    tag,
                    variant=variant,
                    postswap_mode=postswap_mode,
                )
            ):
                print(f"[skip] {tag} ({variant}, postswap={postswap_mode})")
                continue

            print(f"=== [{tag}] ({variant}, postswap={postswap_mode}) ===")
            out = reconstruct_one_tag(
                cfg,
                tag,
                variant=variant,
                postswap_mode=postswap_mode,
            )

            print(
                f"[done] {tag} ({variant}, postswap={postswap_mode}) "
                f"splits={len(out.get('internal_splits', []))} "
                f"fallback={out.get('fallback_count')} "
                f"postswaps={out.get('total_postswaps')} "
                f"solver_calls={out.get('solver_call_count')} "
                f"solver_time={out.get('solver_time_sec_total'):.2f}s "
                f"wall={out.get('reconstruction_wall_time_sec'):.2f}s"
            )
